# Hp30 Probabilistic Forecasting — Implementation Guide

A step-by-step build order for upgrading the regression pipeline from
"percentile-collapsed LogNormal MLP" to "permutation-invariant set encoder with a
BMA-over-members mixture, trained with a proper tail-weighted score."

The guide is ordered by **dependency and information-per-effort**, not by where the
final paper will present things. The governing principle: *do the cheap things that
tell you whether the expensive things are worth doing, first.* Each step is
independently testable and most are independently shippable.

File paths use the package roots:
- `storm_utils/storm_utils/data_structure.py` — the **live** `ForecastingDataset` / `ForecastingConfig` (ignore `old_data_structure.py`)
- `storm_regression/training_pipeline.py` — configs, losses, models, train/eval/driver
- `storm_regression/nn_architectures/` — model classes
- `storm_regression/forecast_analysis.py` — metrics and dashboards
- `storm_regression/case_study_analysis.py` — `load_results`, `recreate_dataset_from_results`, `save_results`
- `results_analysis.ipynb`, `training.ipynb` — notebooks

> **A note on `__getitem__` keys.** Everything below assumes the batch dict keys I saw
> in the dataset (`v`, `vgrad`, `vomni`, `omni`, `omni_sw`, `historic_target`,
> `max_target`, `27_day_target`, `center_idx`). Confirm these against the *live*
> `data_structure.py` before you start — they matched `prepare_mlp_features`, so they're
> almost certainly current, but verify.

---

## Ordering at a glance

```
PHASE 0  Diagnostics (no new model)        ← decides whether Phase 3 is worth building
  0.1  Fix CRPS-object mismatch in dashboard
  0.2  Closed-form LogNormal CRPS
  0.3  PIT histogram
  0.4  "median-only vs full-percentile" delta reader

PHASE 1  Proper tail-weighted loss (twCRPS) ← useful regardless of architecture
  1.1  twCRPS loss + config + wiring

PHASE 2  Latent bug fixes                    ← do before building on top
  2.1  select_best_ensemble_members 3D indexing

PHASE 3  Set encoder + BMA mixture (novelty) ← only if Phase 0 justifies it
  3.1  prepare_set_features (keep the member axis)
  3.2  TemporalConvEncoder
  3.3  SetLogNormalMLP (attention + per-member head)
  3.4  mixture loss (NLL + CRPS/twCRPS)
  3.5  train_set_mlp
  3.6  evaluate_set_mlp_test_set
  3.7  wire model_type='set_mlp' into run_training_pipeline
  3.8  TrainingConfig additions

PHASE 4  Context conditioning & honest CME tail
  4.1  lead-time as input (single model across leads)
  4.2  solar-cycle context features
  4.3  activity-gated tail component

PHASE 5  Evaluation upgrades
  5.1  spread-skill diagram
  5.2  mixture-aware dashboard + PIT
```

Phases 0, 1, 2 are independent of each other and can be done in any order (0 first is
recommended). Phase 3 depends on 1 and 2. Phase 4 depends on 3. Phase 5 can begin as
soon as Phase 3 produces output.

---

## PHASE 0 — Diagnostics first (no new model)


**Why this phase exists.** You already have ~30 trained-result `.pkl` files from your
Phase 1–3 sweeps. Before writing a single line of new model code, those files can answer
the one question that determines everything downstream: *does the ensemble's internal
structure actually carry signal that your percentile features are throwing away?* If it
does, the set encoder is justified. If it doesn't, you've saved yourself weeks and
learned something important about your ensemble (it's under-dispersed, or the target is
median-dominated). This phase is nearly free and decision-critical.

### Step 0.1 — Fix the CRPS-object mismatch in the dashboard

**Where:** `forecast_analysis.py` → `create_probabilistic_dashboard` →
`compute_probabilistic_metrics` (the CRPS block, ~lines 918–933).

**What:** Right now CRPS is computed from `filtered_ensemble` (the raw HUXt-derived
ensemble of point predictions), while the reliability/calibration block (~line 942) uses
the fitted LogNormal (`log_mu`, `log_sigma`). These are **two different forecast
objects.** For the MLP path, your *actual* probabilistic forecast is the LogNormal — so
the CRPS number you're currently reading does not score the model you're proposing.

Replace the sampling-based CRPS-of-the-ensemble with the closed-form CRPS of the
LogNormal (built in 0.2), so CRPS and reliability describe the *same* distribution.

**Why it matters first:** Until this is fixed, every Phase 3 comparison you make is
partly an artefact of which object each metric happened to score. You can't trust the
loss-function sweep conclusions until CRPS and calibration agree on what they're
measuring. This is a correctness fix, not an enhancement.

**Verify:** For one result file, the dashboard CRPS should now change, and it should move
in the *same direction* as calibration quality across your Phase 3 runs (better-calibrated
runs should have lower CRPS). If they still disagree, something else is wrong.

### Step 0.2 — Add closed-form LogNormal CRPS

**Where:** `forecast_analysis.py` → new function `crps_lognormal_closed_form`, sitting
next to the existing `crps_weibull` (~line 65).

**What:** The CRPS of a LogNormal predictive distribution has a closed form (Baran &
Lerch, 2015), so there's no need for Monte Carlo noise when *evaluating*:

```python
import numpy as np
from scipy.stats import norm

def crps_lognormal_closed_form(y, log_mu, log_sigma):
    """
    Closed-form CRPS for a LogNormal predictive distribution.
    Baran & Lerch (2015). All inputs array-like, elementwise. Lower is better.

    y        : observed Hp30max  (> 0)
    log_mu   : predicted mu     (log-space mean)
    log_sigma: predicted sigma  (log-space std, > 0)
    """
    y = np.maximum(np.asarray(y, dtype=float), 1e-9)
    sigma = np.asarray(log_sigma, dtype=float)
    mu = np.asarray(log_mu, dtype=float)

    omega = (np.log(y) - mu) / sigma
    mean_ln = np.exp(mu + 0.5 * sigma**2)          # E[Y] for LogNormal
    crps = ( y * (2.0 * norm.cdf(omega) - 1.0)
             - 2.0 * mean_ln * ( norm.cdf(omega - sigma)
                                 + norm.cdf(sigma / np.sqrt(2.0)) - 1.0 ) )
    return crps
```

**Why:** Two reasons. (1) Evaluation should score the predictive distribution exactly,
not a 1000-sample approximation of it, so run-to-run CRPS differences reflect the model
and not sampling jitter. (2) It's the object 0.1 needs. Keep your *training* CRPS
(`crps_lognormal_loss`) as the sampling version — that one needs to be differentiable and
the MC estimator is fine there; it's only *evaluation* that should be closed-form.

**Verify:** On a batch, the closed-form value should agree with the MC
`crps_lognormal_loss` (per-sample, before the mean) to within MC error (~1–2%) as you
raise `n_samples`. This cross-check confirms both implementations.

### Step 0.3 — Add the PIT histogram

**Where:** `forecast_analysis.py` → new function `plot_pit_histogram(results_path)`;
called from a new cell in `results_analysis.ipynb` under each phase you care about.

**What:** The Probability Integral Transform of each observation under its own predictive
CDF:

```python
from scipy.stats import lognorm
# results from load_results(...)
pit = lognorm.cdf(results['y_test'],
                  s=results['log_sigma'],
                  scale=np.exp(results['log_mu']))
# histogram pit in [0,1] with ~10-20 bins
```

Read the shape: **flat** = calibrated; **∪-shaped** = under-dispersed (predictive
distribution too narrow — events fall in the tails more often than the distribution
expects); **∩-shaped** = over-dispersed; **sloped/asymmetric** = biased (systematic
over- or under-prediction).

**Why:** Your reliability diagrams check *threshold-exceedance* calibration (great for the
traffic light), but they can't tell you whether the LogNormal *shape* is right. PIT does.
This is the single diagnostic most likely to reveal that a single LogNormal is the binding
constraint — I'd expect a ∪-shape on storm-heavy subsets, which is precisely the
under-dispersion the mixture in Phase 3 is designed to fix. If PIT is already flat, the
LogNormal shape is fine and the percentile-collapse (not the distribution family) is your
problem — which still points to the set encoder, but tells you not to bother with the
mixture/GPD-tail machinery.

**Verify:** Produce PIT for "all test data" and for "storms > 4.5" separately. The
contrast between them is the informative bit — a flat all-data PIT with a ∪-shaped
storm PIT is the classic "calibrated on the bulk, under-dispersed on the tail" signature.

### Step 0.4 — "Median-only vs full-percentile" delta reader

**Where:** New cell in `results_analysis.ipynb`, reading your existing
`phase3_ensemble_subsampling_*` result folders with `load_results`.

**What:** You already ran the experiment that answers the central question — you just
haven't framed it this way. Across your `mlp_ensemble_percentiles` sweep (from `[50]` up
to the 11-point sets) and `n_ensemble_keep` sweep, extract the closed-form CRPS (0.2) and
compute: *how much does the best percentile configuration beat `[50]`-only?*

Tabulate CRPS (and threshold-wise BSS) as a function of the number of percentile features,
holding everything else fixed.

**Why this is the linchpin:** Your entire percentile/`n_keep` search space exists *only
because* you are hand-choosing how to summarise the ensemble. A permutation-invariant set
encoder dissolves that axis — it learns whatever ensemble summary matters. So:
- **Large gap (median-only ≪ full percentiles):** the ensemble's internal structure
  carries real signal your features partially capture and the set encoder should capture
  more fully → **build Phase 3.**
- **Small gap:** either the ensemble is under-dispersed (most members agree, so summarising
  them barely helps) or the target is median-dominated → the set encoder will give little,
  and your effort should go to ensemble *quality* (data assimilation, boundary conditions)
  instead. This is itself a publishable finding.

**Verify:** This step produces a single plot/table that you will literally use to decide
whether to proceed to Phase 3. Treat its output as a go/no-go gate.

> **Sub-note worth putting in the paper:** `per_timestep` percentiles
> (`np.percentile(v, p, axis=2)`) construct a velocity curve *no real ensemble member
> produced* — the 95th-percentile-at-every-timestep trajectory is physically impossible
> and destroys within-member temporal coherence (a real high-speed-stream ramp). `snap`
> keeps whole real members. If `snap` beats `per_timestep` in 0.4, that's direct,
> mechanistic evidence for why a member-encoding architecture should win, and it's a clean
> paragraph of physical justification for the set encoder.

---


## PHASE 1 — Proper tail-weighted loss (twCRPS)

**Why this phase exists, and why now.** Your Phase 3 already sweeps
`intensity_strength` from 0.01 to 50, multiplying the per-sample CRPS by an intensity
weight. The problem: multiplying a proper scoring rule by an outcome-dependent weight
*outside* the integral breaks propriety — those runs optimise something subtly improper,
so their calibration is hard to interpret and the "best strength" you pick may just be the
one that games the improper objective. Threshold-weighted CRPS (Gneiting & Ranjan, 2011)
gets the same "care more about big storms" emphasis while staying proper, which keeps your
PIT and reliability plots meaningful. This is architecture-independent — it improves the
*current* MLP and the *future* set encoder identically, so it's worth doing before the big
build.

### Step 1.1 — twCRPS loss, config, and wiring

**Where:**
- `training_pipeline.py` → new function `tw_crps_lognormal_loss` (next to
  `crps_lognormal_loss`, ~line 657)
- `training_pipeline.py` → new dataclass `TwCRPSLossConfig(LossConfig)` (next to
  `CRPSLossConfig`, ~line 60)
- `training_pipeline.py` → `get_loss_function` (~line 76): add an `elif
  loss_config.loss_type == 'twcrps'` bran ch

**What:** The key trick — threshold-weighted CRPS with an upper-tail weight `w(z) = 1{z ≥
t}` equals the *ordinary* CRPS computed on the transformed variable `v(z) = (z − t)_+`
(the chaining function whose derivative is `w`). So you reuse your existing two-sample
energy-score estimator almost verbatim; you just push samples and `y` through `v` first.
Use a softplus for a smooth, differentiable weight:

```python
def tw_crps_lognormal_loss(log_mu_pred, log_sigma_pred, y_true,
                           threshold=4.66, sharpness=2.0, n_samples=1000):
    """
    Threshold-weighted CRPS (upper tail) for a LogNormal, via the chaining
    function v(z) with v'(z) = w(z). Smooth weight: w concentrates mass above
    `threshold`. Proper scoring rule (unlike intensity-multiplied CRPS).
    """
    y_true = torch.clamp(y_true, min=1e-6)
    B = log_mu_pred.shape[0]

    def chain(z):  # v(z) = softplus(sharpness*(z - t)) / sharpness  -> approx (z-t)_+
        return torch.nn.functional.softplus(sharpness * (z - threshold)) / sharpness

    eps  = torch.randn(n_samples, B, device=log_mu_pred.device)
    eps2 = torch.randn(n_samples, B, device=log_mu_pred.device)
    s1 = torch.exp(log_mu_pred.unsqueeze(0) + log_sigma_pred.unsqueeze(0) * eps)
    s2 = torch.exp(log_mu_pred.unsqueeze(0) + log_sigma_pred.unsqueeze(0) * eps2)

    vs1, vs2 = chain(s1), chain(s2)
    vy = chain(y_true).unsqueeze(0)

    term1 = torch.mean(torch.abs(vs1 - vy), dim=0)
    term2 = 0.5 * torch.mean(torch.abs(vs1 - vs2), dim=0)
    return torch.mean(term1 - term2)
```

**Why the chaining trick is the right implementation:** It means twCRPS is a one-line
conceptual change from your existing CRPS (transform, then score), so it's low-risk and
obviously correct by construction. Setting `threshold=4.66` puts the emphasis exactly at
your storm definition; raising it (e.g. 6.0) focuses training on the severe events you most
want to get right. `sharpness` trades off how abruptly the weight switches on (high =
closer to the hard indicator, but noisier gradients).

**Decommission the improper path:** Once twCRPS works, retire the `intensity_*` arguments
from the CRPS loss (keep them on the NLL config if you still use NLL for ablations, but
stop treating intensity-multiplied CRPS as a headline result). Replace the Phase 3
`intensity_strength` sweep with a small `threshold ∈ {4.66, 5.5, 6.5}` sweep.

**Verify:** Train one model each with `crps` and `twcrps`. The twCRPS model should improve
tail metrics (BSS at ≥6, ≥7; storm-subset CRPS) at some cost to bulk CRPS — that trade-off
*is* the point, and it should be visible and monotonic in `threshold`. If twCRPS doesn't
beat plain CRPS on the tail, check that your `threshold` is actually in the data's support
and that storms aren't being dropped upstream.

---



## PHASE 2 — Latent bug fixes

### Step 2.1 — Fix `select_best_ensemble_members` 3D indexing

**Where:** `training_pipeline.py` → `select_best_ensemble_members` (~line 780, the
`v.ndim == 3` branch).

**What:** The line `v[b, :, best_indices[b]]` uses NumPy advanced indexing that transposes
the result to `(n_keep, T_full)`, so the subsequent `np.stack(..., axis=0)` yields
`(B, n_keep, T_full)` — **not** the `(B, T_full, n_keep)` the docstring and downstream code
assume. Fix:

```python
v_filtered = np.stack([v[b][:, best_indices[b]] for b in range(B)], axis=0)
# v[b] is (T_full, N_ens); [:, idx] keeps (T_full, n_keep) correctly
```

**Why before Phase 3:** Any `filter_ensemble=True` run in your Phase 3 subsampling sweeps
has been silently feeding mis-shaped arrays into percentile extraction. If those runs
looked oddly bad or noisy, this is why — and you don't want to carry a silent shape bug
into the set encoder, which also consumes the filtered ensemble. Re-run any
`filter_ensemble=True` conclusions after fixing.

**Verify:** Add an `assert v_filtered.shape == (B, T_full, n_keep)` immediately after the
stack. Re-run one filtered Phase 3 config and confirm the metrics change.

---

## PHASE 3 — Set encoder + BMA mixture (the core novelty)



**Why this phase, and the one-sentence summary of the idea.** Your current
`prepare_mlp_features` reduces the `(B, T, Nens)` ensemble to a few per-timestep percentile
*curves* before the network sees it — so `LogNormalMLP` never sees *members*, only
envelopes, and it cannot represent bimodality (storm / no-storm timing splits) or exploit
the link between "which member tracked OMNI historically" and "what that member predicts."
The set encoder keeps the member axis, encodes each member with a shared temporal network,
learns an attention weight per member (the principled replacement for your `1/MAE²`
heuristic), and produces a **mixture of LogNormals** — one component per member, weighted by
attention. That is neural Bayesian Model Averaging over the HUXt ensemble.

> Build this **only if Step 0.4 showed a meaningful median-only-vs-percentile gap.**

### Step 3.1 — `prepare_set_features` (keep the member axis)

**Where:** `training_pipeline.py` → new function `prepare_set_features`, parallel to
`prepare_mlp_features` (~line 798).

**What:** Return a per-member tensor and a shared-context tensor, *without* flattening the
member axis:

```python
def prepare_set_features(batch, n_ensembles, scaler=None, fit_scaler=False):
    v     = batch['v'].numpy()        # (B, T_full, Nens)
    vgrad = batch['vgrad'].numpy()    # (B, T_full, Nens)
    vomni = batch['vomni'].numpy()    # (B, T_input, Nens)  per-member v - OMNI
    omni  = batch['omni'].numpy()     # (B, T_input, N_omni)  shared context
    hist  = batch['historic_target'].numpy()  # (B, T_input, 1) shared
    omni_sw = batch['omni_sw'].numpy()         # (B, T_input)  for warm-start MAE

    B, T_full, M = v.shape
    T_in = vomni.shape[1]

    # Per-member channels, aligned on the full time axis.
    # vomni only exists in the input window -> pad the forecast region with 0
    # and carry a mask channel so the encoder knows which steps are observed.
    pad = T_full - T_in
    vomni_full = np.concatenate([vomni, np.zeros((B, pad, M))], axis=1)
    mask = np.concatenate([np.ones((B, T_in, M)), np.zeros((B, pad, M))], axis=1)

    # member tensor: (B, M, C, T_full), C = [v, vgrad, vomni, mask]
    members = np.stack([v, vgrad, vomni_full, mask], axis=2).transpose(0, 3, 2, 1)

    # per-member input-window MAE vs OMNI V_sw  -> warm-start / attention feature
    member_mae = np.mean(np.abs(v[:, :T_in, :] - omni_sw[:, :, None]), axis=1)  # (B, M)

    # shared context: flattened omni + historic Hp30 (these are NOT per-member)
    context = np.concatenate([omni.reshape(B, -1), hist.reshape(B, -1)], axis=1)

    # scale (fit MinMax on members and context separately; store both in scaler)
    ...
    return members, member_mae, context, scaler
```

**Why this shape:** `(B, M, C, T)` lets Step 3.2 apply one shared encoder to all `B*M`
members in a single reshaped forward pass — efficient and *permutation-invariant by
construction* (the same weights see every member). The `mask` channel is important: `vomni`
is only defined in the input window, and without a mask the encoder can't distinguish "zero
difference" from "no observation," which would corrupt the forecast-window encoding. The
`member_mae` is the exact quantity your `calculate_ensemble_weights` already computes — you
pass it to the attention head so the network can *start* from your `1/MAE²` intuition and
improve on it.

**Verify:** Shapes. `members.shape == (B, M, 4, T_full)`, `member_mae.shape == (B, M)`,
`context.shape == (B, context_size)`. Then confirm that averaging `members[:, :, 0, :]` over
the member axis reproduces the ensemble-mean velocity you'd get the old way.

### Step 3.2 — `TemporalConvEncoder`

**Where:** new file `storm_regression/nn_architectures/temporal_conv_encoder.py`.

**What:** A small dilated 1-D CNN that maps one member's `(C, T)` channels to an embedding
`z_m`. This is essentially your existing `ConvEncoder` (from `conv_forecast_regressor.py`)
with dilations added so the receptive field spans the relevant solar-wind timescales:

```python
import torch.nn as nn

class TemporalConvEncoder(nn.Module):
    def __init__(self, in_channels, embed=64, kernel=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel, padding=1, dilation=1), nn.ReLU(),
            nn.Conv1d(32, 64, kernel, padding=2, dilation=2),          nn.ReLU(),
            nn.Conv1d(64, embed, kernel, padding=4, dilation=4),       nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
    def forward(self, x):          # x: (N, C, T)
        return self.net(x).squeeze(-1)   # (N, embed)
```

**Why a dilated CNN rather than GRU/transformer:** The physically meaningful, v-only
signal for the storms you *can* forecast is the *shape of the velocity ramp* — the stream
interface / compression at the leading edge of a high-speed stream that drives CIR storms.
CNNs are excellent ramp/edge detectors, dilations give a multi-timescale receptive field
cheaply, and the architecture is small enough to apply to `B*M` members per batch without
blowing up compute. You're literally reusing the encoder you already wrote and validated
in the classification work; only the dilation and the head differ.

**Verify:** Feed `members.reshape(B*M, C, T)` and confirm output `(B*M, embed)`.

### Step 3.3 — `SetLogNormalMLP` (attention + per-member head)

**Where:** new file `storm_regression/nn_architectures/set_lognormal_mlp.py`.

**What:**

```python
import torch
import torch.nn as nn
from .temporal_conv_encoder import TemporalConvEncoder

class SetLogNormalMLP(nn.Module):
    """Permutation-invariant set -> BMA-over-members LogNormal mixture."""
    def __init__(self, member_channels, context_size, embed=64):
        super().__init__()
        self.encoder = TemporalConvEncoder(member_channels, embed)
        self.attn = nn.Linear(embed + 1, 1)          # +1 = per-member input MAE
        self.head = nn.Sequential(
            nn.Linear(embed + context_size, 64), nn.ReLU(),
            nn.Linear(64, 2),                         # (mu_m, raw_sigma_m)
        )
        self.softplus = nn.Softplus()

    def forward(self, members, member_mae, context):
        B, M, C, T = members.shape
        z = self.encoder(members.reshape(B * M, C, T)).reshape(B, M, -1)   # (B,M,E)
        logits = self.attn(torch.cat([z, member_mae.unsqueeze(-1)], -1)).squeeze(-1)
        alpha = torch.softmax(logits, dim=1)                                # (B,M)
        ctx = context.unsqueeze(1).expand(-1, M, -1)
        out = self.head(torch.cat([z, ctx], -1))                            # (B,M,2)
        mu_m = out[..., 0]
        sigma_m = self.softplus(out[..., 1]) + 0.01
        return mu_m, sigma_m, alpha   # mixture: sum_m alpha_m LogNormal(mu_m, sigma_m)
```

**Why each piece:**
- **Shared `encoder` over `B*M`** is what makes the model permutation-invariant and
  parameter-efficient, and it lets you train on 100 members and deploy on 200 (or fewer)
  without retraining — operationally valuable.
- **`attn` conditioned on `member_mae`** is the learned generalisation of your `1/MAE²`
  weighting. The network can recover inverse-error weighting if that's optimal, but it can
  also down-weight a well-aligned member whose *forecast-window* trajectory looks
  implausible — something the hand-coded heuristic can't do.
- **Per-member `(mu_m, sigma_m)`** gives the mixture its components. The predictive
  distribution `Σ_m alpha_m LogNormal(mu_m, sigma_m)` is multimodal-capable, so it can
  represent "60% of members say no storm, 40% say a strong one" instead of collapsing to a
  fat unimodal blob at the mean — directly fixing the indecisiveness you documented on
  large events.
- **Variance decomposition for free:** within-member `sigma_m` = aleatoric uncertainty
  given a solar-wind scenario; between-member spread of `mu_m` = ensemble/scenario
  uncertainty. Report both per lead time — it's an independently interesting result and
  the honest basis for the CME-tail story in Phase 4.

**Verify:** `mu_m, sigma_m, alpha` shapes all `(B, M)` (alpha rows sum to 1). Exceedance:
`P(Y≥θ) = Σ_m alpha_m (1 − Φ((lnθ − mu_m)/sigma_m))`.

### Step 3.4 — Mixture loss (NLL and CRPS/twCRPS)

**Where:** `training_pipeline.py` → new `mixture_nll_loss` and `mixture_crps_loss`; new
`get_loss_function` branches; the loss signature changes from `(log_mu, log_sigma, y)` to
`(mu_m, sigma_m, alpha, y)` for the set path.

**What:**
- **Mixture NLL:** `−log Σ_m alpha_m LogNormal_pdf(y; mu_m, sigma_m)` — use
  `torch.logsumexp(log(alpha) + lognormal_logpdf_m, dim=1)` for numerical stability.
- **Mixture (tw)CRPS:** sample a member index from `alpha`, then a LogNormal draw from that
  member, and apply your existing two-sample energy-score estimator (and the chaining
  transform from 1.1 for the tail-weighted version). Sampling the categorical via
  Gumbel-softmax or a straight-through estimator keeps `alpha` differentiable.

**Why a new signature instead of reusing the old one:** The mixture genuinely has more
parameters per sample, and forcing it through the `(log_mu, log_sigma)` interface would
lose the per-member structure that's the whole point. Keep both loss families: NLL is the
fast, stable default for getting the architecture training; (tw)CRPS is the headline,
calibration-friendly objective for final results. Use `logsumexp` — naive
`log(sum(exp))` will underflow on quiet windows where all components put tiny density at
the observed low Hp30.

**Verify:** On a degenerate mixture (set `alpha` to one-hot, all components equal), the
mixture losses must reduce *exactly* to your single-LogNormal `negative_log_likelihood_*`
and `crps_lognormal_loss`. This is the unit test that proves the mixture code is correct.

### Step 3.5 — `train_set_mlp`

**Where:** `training_pipeline.py` → new function mirroring `train_mlp` (~line 890).

**What:** Same skeleton as `train_mlp` (optimizer, early stopping, scaler-fit-on-first-
batch), but calls `prepare_set_features`, instantiates `SetLogNormalMLP`, and uses the
mixture loss. Pass `member_mae` through to the forward call.

**Why mirror rather than generalise `train_mlp`:** Keep them separate while iterating —
you don't want a refactor of the working MLP path to be a prerequisite for experimenting
with the set encoder. Unify later once the set path is proven.

**Verify:** It runs one epoch on a small `Subset` without shape errors and the loss
decreases over a few epochs on a tiny overfit set (10 windows).

### Step 3.6 — `evaluate_set_mlp_test_set`

**Where:** `training_pipeline.py` → new function mirroring `evaluate_mlp_test_set`
(~line 1004).

**What:** Emit the mixture parameters plus derived quantities so the rest of the pipeline
keeps working:
- `mu_m`, `sigma_m`, `alpha` (the full mixture — new)
- `y_pred_mixture_median` (numerically: invert the mixture CDF, or use the
  attention-weighted member medians as a fast proxy) — fills the role of
  `y_pred_lognormal_median`
- `exceedance_probs` at your traffic-light thresholds (4.66, 6.5, 8) — the operational
  output
- the existing baselines (`persistence`, `persistence_max`, `27_day_recurrence`) unchanged

**Why preserve the output contract:** `save_results`, `MetricWriter`, and your dashboards
all key off these names. If the set path emits a superset of the MLP path's keys, your
entire `results_analysis.ipynb` keeps working with minimal edits — you only add
mixture-specific plots in Phase 5.

**Verify:** Run `auto_detect_and_compare` over a set-encoder result file and confirm it
loads and tabulates without errors.

### Step 3.7 — Wire `model_type='set_mlp'` into the driver

**Where:** `training_pipeline.py` → `run_training_pipeline`, the two `if config.model_type
== 'mlp'` blocks (training-side and evaluation-side, ~lines 1435 and 1485).

**What:** Add an `elif config.model_type == 'set_mlp':` branch in both places, calling
`train_set_mlp` / `evaluate_set_mlp_test_set`. In the metrics-aggregation block, route the
`'mixture_median'` aggregator to the mixture CRPS (closed-form per component, weighted).

**Why last among the 3.x build steps:** The driver is the integration point; wiring it
before the pieces exist just produces import errors. Once 3.1–3.6 each pass their own
verify step, this is a ~20-line branch addition and your existing CV / seed / fold loops,
`save_results`, and `MetricWriter` all carry over unchanged.

**Verify:** `config = TrainingConfig(model_type='set_mlp', ...)` then
`run_training_pipeline(config)` completes end-to-end on `test_folds=[0]`,
`random_seeds=[42]`, one lead time, and writes a result `.pkl`.

### Step 3.8 — `TrainingConfig` additions

**Where:** `training_pipeline.py` → `TrainingConfig` dataclass (~line 120).

**What:** Add `set_embed_dim: int = 64`, `set_encoder_kernel: int = 3`, and a
`SetCRPSLossConfig` / `SetNLLLossConfig` if you want set-specific loss defaults. Keep all
the existing fields — the set path ignores `mlp_ensemble_percentiles`,
`ensemble_selection_method`, etc., which is fine (and a nice demonstration that those knobs
are obsoleted by the architecture).

**Verify:** Defaults instantiate; a set-encoder run reads them.

---



## PHASE 4


**Why now and not earlier:** These are refinements that only make sense once the set
encoder works and you've measured its baseline behaviour. Adding them earlier would
confound "does the architecture help" with "does extra context help."

### Step 4.1 — Lead time as an input (one model across all leads)

**Where:** `data_structure.py` (`__getitem__` — add `lead_time` to the returned dict, or
pass it via config); `prepare_set_features` (append normalised lead time to `context`);
`SetLogNormalMLP` (no change — `context_size` grows by one).

**Why:** Paper 1 and your current runs train a separate model per lead time. Conditioning
one model on lead time is far more data-efficient, gives smooth behaviour across the
horizon, and lets the network learn the handover you already documented — from
Hp30-history-dominated at short lead to ensemble-dominated at long lead — *inside one
model* rather than as a discontinuous jump between separately-trained ones. To train it,
sample lead time per batch (or per window) across your range of interest.

**Verify:** Train one model with lead time sampled in `{1,3,6,12,24,36}`h; its
per-lead-time CRPS should be competitive with the lead-time-specific models, and the
lead-time→skill curve should be smooth.

### Step 4.2 — Solar-cycle context features

**Where:** `prepare_set_features` — pull `f107` and/or `sunspot_R` from the OMNI columns
(they're already in `all_omni_columns`) into `context`.

**Why:** It conditions the bulk-vs-tail mixing (CME-driven extremes are far likelier near
solar max) and it's *why* 27-day recurrence beats you at solar minimum — give the model
the information explicitly rather than making it infer cycle phase from velocity. Cheap;
the data is already loaded.

**Verify:** Ablate with/without; expect the largest improvement on high-threshold
calibration during active periods.

### Step 4.3 — Activity-gated tail component

**Where:** `SetLogNormalMLP` — add one extra, deliberately heavy-tailed mixture component
whose weight is gated by the solar-activity context from 4.2.

**Why:** HUXt-ambient cannot see CMEs — your FN case proved a Bz-driven storm at modest
velocity is unforecastable from velocity ensembles. The honest response is not to chase it
with features but to *widen the predictive tail* when CMEs are climatologically likely, so
the forecast says "we can't predict the transient, but the risk of a big surprise is
elevated." This converts your single biggest limitation into a calibrated feature of the
probabilistic output, and it's the natural use of the within/between variance decomposition
from 3.3.

**Verify:** With `remove_cmes=True` vs `False`, show the model cannot catch removed CMEs
(expected) but that the gated tail keeps the high-threshold PIT closer to flat during
active periods than the un-gated mixture.

---



## PHASE 5 — Evaluation upgrades

### Step 5.1 — Spread-skill diagram

**Where:** `forecast_analysis.py` → new function; cell in `results_analysis.ipynb`.

**What:** Bin forecasts by predicted spread (mixture std, decomposable into within/between
from 3.3) and plot mean predicted spread vs actual RMSE per bin.

**Why:** This is the diagnostic the ensemble-postprocessing community will demand: it
proves you're using the ensemble's dispersion *correctly* — that when the ensemble
disagrees, the model widens, and that the widening tracks real error. A flat or inverted
spread-skill line means the attention/variance machinery isn't earning its place.

### Step 5.2 — Mixture-aware dashboard + PIT

**Where:** `forecast_analysis.py` → extend `create_probabilistic_dashboard` to accept the
mixture parameters; reuse the PIT from 0.3 but with the **mixture CDF**
(`Σ_m alpha_m lognorm.cdf(...)`) instead of the single LogNormal.

**Why:** Closes the loop — the same calibration story you told for the single LogNormal in
Phase 0, now for the mixture, so you can show *directly* that the mixture flattens the
∪-shaped storm-tail PIT that motivated building it. That before/after PIT comparison is
likely your single most persuasive figure for the methods section.

---

## Suggested commit checkpoints

Each of these is a self-contained, reviewable unit of work that leaves the pipeline in a
runnable state:

1. `fix: score CRPS and reliability on the same LogNormal object` (0.1 + 0.2)
2. `feat: PIT histogram + median-vs-percentile diagnostic` (0.3 + 0.4) ← **go/no-go gate**
3. `feat: threshold-weighted CRPS loss` (1.1)
4. `fix: correct 3D indexing in select_best_ensemble_members` (2.1)
5. `feat: set-feature prep preserving me§mber axis` (3.1)
6. `feat: temporal conv encoder + set LogNormal mixture model` (3.2 + 3.3)
7. `feat: mixture losses with single-LogNormal reduction test` (3.4)
8. `feat: train/eval/driver wiring for model_type=set_mlp` (3.5–3.8)
9. `feat: lead-time + solar-cycle conditioning` (4.1 + 4.2)
10. `feat: activity-gated CME tail component` (4.3)
11. `feat: spread-skill + mixture-aware calibration dashboards` (5.x)

The decision gate is checkpoint 2. Everything from 5 onward is contingent on it showing
that the ensemble's internal structure carries signal worth recovering.

---

## References for the methods section

- **EMOS / nonhomogeneous regression:** Gneiting, Raftery, Westveld & Goldman (2005).
- **Bayesian Model Averaging of ensembles:** Raftery, Gneiting, Balabdaoui & Polakowski (2005).
- **Neural ensemble postprocessing (DRN, CRPS loss):** Rasp & Lerch (2018).
- **Mixture density networks:** Bishop (1994).
- **Deep Sets (permutation-invariant set functions):** Zaheer et al. (2017).
- **Threshold-weighted CRPS:** Gneiting & Ranjan (2011).
- **Closed-form LogNormal CRPS:** Baran & Lerch (2015).

Verify exact citation details before submission — these are from memory and the field
moves; a quick literature check on permutation-invariant postprocessing in space weather
specifically will also help you position the novelty cleanly.